In [1]:
from google.colab import files
import pandas as pd

# Abre el explorador de archivos local
uploaded = files.upload()

Saving Dataset_consolidado_final_v4.csv to Dataset_consolidado_final_v4.csv


In [4]:
# -*- coding: utf-8 -*-

"""
SISTEMA HIBRIDO:
LIGHTGBM + DEEPSEEK API

Incluye:
- LightGBM
- Exportacion de modelo:
    * joblib
    * pickle
    * txt
    * json
    * bin
- DeepSeek prompting:
    * Zero-shot
    * Few-shot
    * Chain of Thought (CoT)
- Ensemble hibrido:
    * 80% LightGBM
    * 20% DeepSeek

CORRECCIONES:
- NO usa train_test_split
- Usa Dataset_type:
    * TRAIN
    * TEST
    * EVAL
- Entrena SOLO con TRAIN
- Evalua TEST y EVAL
- DeepSeek usa EVAL
- Mantiene 30 ejemplos prompting
"""

# =========================================================
# 0. INSTALAR LIBRERIAS (SOLO COLAB)
# =========================================================

# !pip install lightgbm openpyxl -q

# =========================================================
# 1. LIBRERIAS
# =========================================================

import pandas as pd
import numpy as np
import requests
import scipy.sparse as sp
import time
import json
import pickle
import joblib

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

from sklearn.feature_extraction.text import TfidfVectorizer

import lightgbm as lgb

# =========================================================
# 2. CARGAR DATASET
# =========================================================

data = pd.read_csv(
    "Dataset_consolidado_final_v4.csv"
)

# =========================================================
# 3. LIMPIEZA BASICA
# =========================================================

data = data.dropna(
    subset=[
        "text",
        "label",
        "dataset_type"
    ]
)

data["text"] = data["text"].astype(str)

data["dataset_type"] = (
    data["dataset_type"]
    .astype(str)
    .str.upper()
    .str.strip()
)

# =========================================================
# 4. LABEL BINARIO
# =========================================================
# =========================================================
# 4. LABEL MULTICLASE
# =========================================================

from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

data["label"] = label_encoder.fit_transform(
    data["label"].astype(str)
)

NUM_CLASSES = len(label_encoder.classes_)

print("\nClases detectadas:")
print(label_encoder.classes_)
print("Numero de clases:", NUM_CLASSES)

# =========================================================
# 5. DIVISIONES DATASET
# =========================================================

train_data = data[
    data["dataset_type"] == "TRAIN"
]

test_data = data[
    data["dataset_type"] == "TEST"
]

eval_data = data[
    data["dataset_type"] == "EVAL"
]

print("\n===================================")
print("TRAIN")
print("===================================")
print(len(train_data))

print("\n===================================")
print("TEST")
print("===================================")
print(len(test_data))

print("\n===================================")
print("EVAL")
print("===================================")
print(len(eval_data))

# =========================================================
# 6. TF-IDF
# =========================================================

vectorizer = TfidfVectorizer(
    max_features=5000
)

X_train_text = vectorizer.fit_transform(
    train_data["text"]
)

X_test_text = vectorizer.transform(
    test_data["text"]
)

X_eval_text = vectorizer.transform(
    eval_data["text"]
)

# =========================================================
# 8. MATRICES FINALES
# =========================================================

X_train = sp.hstack([
    X_train_text
])

X_test = sp.hstack([
    X_test_text
])

X_eval = sp.hstack([
    X_eval_text
])

y_train = train_data["label"]

y_test = test_data["label"]

y_eval = eval_data["label"]

# =========================================================
# 9. FUNCION METRICAS (MULTICLASE)
# =========================================================

def evaluar_modelo(
    nombre,
    y_true,
    y_pred,
    y_proba=None
):

    print("\n================================================")
    print(nombre)
    print("================================================")

    print(
        "Accuracy :",
        round(
            accuracy_score(y_true, y_pred),
            4
        )
    )

    print(
        "Precision (macro):",
        round(
            precision_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),
            4
        )
    )

    print(
        "Recall (macro)   :",
        round(
            recall_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),
            4
        )
    )

    print(
        "F1 Score (macro) :",
        round(
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),
            4
        )
    )

    # ROC AUC multiclass
    if y_proba is not None:

        try:

            roc = roc_auc_score(
                y_true,
                y_proba,
                multi_class="ovr",
                average="macro"
            )

            print(
                "ROC-AUC (macro):",
                round(roc, 4)
            )

        except:
            print("ROC-AUC no disponible")

    print("\nReporte completo:\n")

    print(
        classification_report(
            y_true,
            y_pred,
            target_names=label_encoder.classes_,
            zero_division=0
        )
    )
# =========================================================
# 10. LIGHTGBM
# =========================================================

print("\nEntrenando LightGBM...")

lgb_model = lgb.LGBMClassifier(

    objective="multiclass",
    num_class=NUM_CLASSES,

    random_state=42,
    n_estimators=300,
    learning_rate=0.05
)

lgb_model.fit(
    X_train,
    y_train
)

# =========================================================
# 11. EVALUACION LIGHTGBM TEST
# =========================================================

print("\n===================================")
print("LIGHTGBM TEST")
print("===================================")

y_pred_test = lgb_model.predict(
    X_test
)

y_proba_test = lgb_model.predict_proba(
    X_test
)

evaluar_modelo(
    "LightGBM TEST",
    y_test,
    y_pred_test,
    y_proba_test
)

# =========================================================
# 12. EVALUACION LIGHTGBM EVAL
# =========================================================

print("\n===================================")
print("LIGHTGBM EVAL")
print("===================================")

y_pred_eval = lgb_model.predict(
    X_eval
)

y_proba_eval = lgb_model.predict_proba(
    X_eval
)

evaluar_modelo(
    "LightGBM EVAL",
    y_eval,
    y_pred_eval,
    y_proba_eval
)

# =========================================================
# 13. EXPORTAR MODELO
# =========================================================

print("\nExportando modelos...")

joblib.dump(
    lgb_model,
    "lightgbm_model.joblib"
)

with open(
    "lightgbm_model.pkl",
    "wb"
) as f:

    pickle.dump(
        lgb_model,
        f
    )

lgb_model.booster_.save_model(
    "lightgbm_model.txt"
)

lgb_model.booster_.save_model(
    "lightgbm_model.bin"
)

modelo_json = lgb_model.booster_.dump_model()

with open(
    "lightgbm_model.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        modelo_json,
        f,
        indent=4
    )

joblib.dump(
    vectorizer,
    "tfidf_vectorizer.joblib"
)

print("\nModelos exportados correctamente.")


Clases detectadas:
['BACK' 'CONC' 'CONTR' 'DISC' 'INTRO' 'LIM' 'METH' 'RES']
Numero de clases: 8

TRAIN
10761

TEST
4573

EVAL
1699

Entrenando LightGBM...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.032878 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 392048
[LightGBM] [Info] Number of data points in the train set: 10761, number of used features: 4997
[LightGBM] [Info] Start training from score -1.998863
[LightGBM] [Info] Start training from score -2.143231
[LightGBM] [Info] Start training from score -2.040886
[LightGBM] [Info] Start training from score -2.040886
[LightGBM] [Info] Start training from score -2.048785
[LightGBM] [Info] Start training from score -2.119737
[LightGBM] [Info] Start training from score -2.138488
[LightGBM] [Info] Start training from score -2.115104

LIGHTGBM TEST


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



LightGBM TEST
Accuracy : 0.714
Precision (macro): 0.7155
Recall (macro)   : 0.7116
F1 Score (macro) : 0.7124
ROC-AUC (macro): 0.9557

Reporte completo:

              precision    recall  f1-score   support

        BACK       0.73      0.77      0.75       622
        CONC       0.73      0.77      0.75       538
       CONTR       0.78      0.69      0.73       598
        DISC       0.77      0.79      0.78       600
       INTRO       0.81      0.75      0.78       591
         LIM       0.71      0.63      0.67       524
        METH       0.62      0.65      0.63       543
         RES       0.58      0.65      0.61       557

    accuracy                           0.71      4573
   macro avg       0.72      0.71      0.71      4573
weighted avg       0.72      0.71      0.71      4573


LIGHTGBM EVAL


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



LightGBM EVAL
Accuracy : 0.5474
Precision (macro): 0.543
Recall (macro)   : 0.5489
F1 Score (macro) : 0.5167
ROC-AUC (macro): 0.8947

Reporte completo:

              precision    recall  f1-score   support

        BACK       0.66      0.78      0.72       226
        CONC       0.44      0.16      0.23       200
       CONTR       0.45      0.61      0.52       192
        DISC       0.59      0.68      0.63       229
       INTRO       0.64      0.27      0.38       259
         LIM       0.53      0.34      0.41       184
        METH       0.57      0.91      0.70       202
         RES       0.46      0.64      0.54       207

    accuracy                           0.55      1699
   macro avg       0.54      0.55      0.52      1699
weighted avg       0.55      0.55      0.52      1699


Exportando modelos...

Modelos exportados correctamente.

EVALUACION HIBRIDA EVAL
Procesando 1/1699


...

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Procesando 1699/1699


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Hybrid LightGBM + DeepSeek
Accuracy : 0.0


ValueError: Mix of label input types (string and number)

In [ ]:

# =========================================================
# 14. 30 EJEMPLOS PROMPTING
# =========================================================

PROMPT_EXAMPLES = {

# ======================================================
# ZERO SHOT
# ======================================================
"zero-shot": {

"ejemplos": [

# ---------- CONTR (5) ----------
{"texto":"los resultados obtenidos contradicen la hipotesis inicial debido a la ausencia de relacion significativa entre las variables","label":"CONTR"},
{"texto":"a diferencia de estudios previos el modelo propuesto no mejoro el rendimiento del sistema evaluado","label":"CONTR"},
{"texto":"los hallazgos indican que la intervencion aplicada no produjo el efecto esperado en la poblacion analizada","label":"CONTR"},
{"texto":"el analisis estadistico refuta la teoria planteada inicialmente sobre la eficacia del tratamiento","label":"CONTR"},
{"texto":"los datos experimentales muestran resultados opuestos a los reportados en investigaciones anteriores","label":"CONTR"},

# ---------- INTRO (2) ----------
{"texto":"la inteligencia artificial ha emergido como una herramienta clave para el analisis de grandes volumenes de datos cientificos","label":"INTRO"},
{"texto":"el incremento de enfermedades cronicas ha motivado nuevas investigaciones orientadas a mejorar los procesos diagnosticos","label":"INTRO"},

# ---------- BACK (2) ----------
{"texto":"estudios anteriores demostraron que los modelos predictivos mejoran la precision diagnostica en contextos clinicos","label":"BACK"},
{"texto":"investigaciones previas reportaron asociaciones entre factores ambientales y desarrollo de enfermedades metabolicas","label":"BACK"},

# ---------- METHO (2) ----------
{"texto":"se aplico validacion cruzada estratificada para evaluar el rendimiento del modelo propuesto","label":"METHO"},
{"texto":"los participantes fueron evaluados mediante analisis estadistico multivariado y pruebas controladas","label":"METHO"},

# ---------- RES (2) ----------
{"texto":"el modelo alcanzo una precision superior al noventa por ciento en el conjunto de prueba","label":"RES"},
{"texto":"los resultados evidenciaron una reduccion significativa del error respecto a metodos tradicionales","label":"RES"},

# ---------- DISC (2) ----------
{"texto":"estos resultados sugieren que el enfoque propuesto podria mejorar la eficiencia del sistema evaluado","label":"DISC"},
{"texto":"la mejora observada podria explicarse por la capacidad del modelo para capturar patrones complejos","label":"DISC"},

# ---------- CONC (2) ----------
{"texto":"en conclusion el metodo propuesto demuestra ser viable para aplicaciones reales en analisis cientifico","label":"CONC"},
{"texto":"los hallazgos respaldan el uso de modelos inteligentes como apoyo a la toma de decisiones","label":"CONC"},

# ---------- LIM (2) ----------
{"texto":"una limitacion del estudio es el tamaño reducido de la muestra analizada","label":"LIM"},
{"texto":"los resultados deben interpretarse con cautela debido a la falta de diversidad en los datos utilizados","label":"LIM"}

]
},

# ======================================================
# FEW SHOT
# ======================================================
"few-shot": {

"ejemplos": [

# ---------- CONTR (5) ----------
{"texto":"contrario a lo esperado el modelo implementado no genero mejoras en la prediccion de resultados","label":"CONTR"},
{"texto":"los resultados experimentales no respaldan la hipotesis planteada inicialmente","label":"CONTR"},
{"texto":"el estudio evidencio que la estrategia aplicada no tuvo impacto significativo","label":"CONTR"},
{"texto":"la intervencion propuesta no logro mejorar los indicadores evaluados","label":"CONTR"},
{"texto":"los hallazgos contradicen investigaciones previas al mostrar un efecto inverso","label":"CONTR"},

# ---------- INTRO (2) ----------
{"texto":"en los ultimos años el analisis automatico de textos cientificos ha cobrado gran relevancia","label":"INTRO"},
{"texto":"el crecimiento exponencial de datos ha impulsado el desarrollo de tecnicas avanzadas de analisis","label":"INTRO"},

# ---------- BACK (2) ----------
{"texto":"trabajos anteriores han demostrado la utilidad del aprendizaje automatico en tareas de clasificacion","label":"BACK"},
{"texto":"investigaciones previas identificaron beneficios del uso de modelos predictivos en salud","label":"BACK"},

# ---------- METHO (2) ----------
{"texto":"se entreno un modelo utilizando particiones estratificadas del conjunto de datos","label":"METHO"},
{"texto":"se aplicaron tecnicas de preprocesamiento y normalizacion antes del entrenamiento","label":"METHO"},

# ---------- RES (2) ----------
{"texto":"los experimentos mostraron mejoras consistentes en todas las metricas evaluadas","label":"RES"},
{"texto":"se obtuvo un incremento significativo en la exactitud del modelo propuesto","label":"RES"},

# ---------- DISC (2) ----------
{"texto":"los resultados obtenidos permiten interpretar que el modelo captura relaciones complejas","label":"DISC"},
{"texto":"estos hallazgos coinciden parcialmente con investigaciones reportadas anteriormente","label":"DISC"},

# ---------- CONC (2) ----------
{"texto":"el estudio confirma la viabilidad del enfoque propuesto para futuras investigaciones","label":"CONC"},
{"texto":"se concluye que la metodologia desarrollada aporta mejoras relevantes al campo analizado","label":"CONC"},

# ---------- LIM (2) ----------
{"texto":"una limitacion relevante es la dependencia de datos provenientes de una sola fuente","label":"LIM"},
{"texto":"el tamaño del conjunto de entrenamiento podria afectar la generalizacion del modelo","label":"LIM"}

]
},

# ======================================================
# CHAIN OF THOUGHT
# ======================================================
"cot": {

"ejemplos": [

# ---------- CONTR (5) ----------
{"texto":"tras analizar los datos se observo ausencia de diferencias significativas por lo que la hipotesis fue rechazada","label":"CONTR"},
{"texto":"aunque se esperaba una mejora los resultados mostraron un desempeño inferior al metodo base","label":"CONTR"},
{"texto":"el razonamiento derivado del analisis indica que el modelo teorico no coincide con la evidencia empirica","label":"CONTR"},
{"texto":"el analisis comparativo evidencio que la intervencion no produjo cambios relevantes","label":"CONTR"},
{"texto":"despues de evaluar los resultados se concluyo que las medidas aplicadas no fueron efectivas","label":"CONTR"},

# ---------- INTRO (2) ----------
{"texto":"primero se describe el contexto general del problema relacionado con el analisis automatico de informacion cientifica","label":"INTRO"},
{"texto":"se introduce la problematica asociada al crecimiento acelerado de literatura academica","label":"INTRO"},

# ---------- BACK (2) ----------
{"texto":"posteriormente se revisan estudios previos que han abordado problemas similares","label":"BACK"},
{"texto":"la literatura existente muestra avances importantes en modelos de clasificacion automatica","label":"BACK"},

# ---------- METHO (2) ----------
{"texto":"luego se aplico un proceso sistematico de entrenamiento y validacion del modelo","label":"METHO"},
{"texto":"se utilizaron tecnicas de evaluacion cruzada para garantizar la robustez experimental","label":"METHO"},

# ---------- RES (2) ----------
{"texto":"los resultados obtenidos mostraron mejoras consistentes respecto a enfoques tradicionales","label":"RES"},
{"texto":"el analisis final evidencio una reduccion considerable del error de prediccion","label":"RES"},

# ---------- DISC (2) ----------
{"texto":"estos resultados permiten interpretar el impacto del modelo dentro del contexto estudiado","label":"DISC"},
{"texto":"el comportamiento observado puede explicarse por la capacidad adaptativa del algoritmo","label":"DISC"},

# ---------- CONC (2) ----------
{"texto":"finalmente se concluye que el enfoque desarrollado representa una alternativa prometedora","label":"CONC"},
{"texto":"el estudio demuestra el potencial de integrar inteligencia artificial en analisis cientifico","label":"CONC"},

# ---------- LIM (2) ----------
{"texto":"una limitacion identificada es la disponibilidad limitada de datos etiquetados","label":"LIM"},
{"texto":"los resultados podrian variar al aplicarse en contextos diferentes al analizado","label":"LIM"}

]
}

}
# =========================================================
# 15. API DEEPSEEK
# =========================================================

API_KEY = "sk-427c01e03b854dbbae3b791e765f9a65"

API_URL = "https://api.deepseek.com/chat/completions"

headers = {

    "Authorization": f"Bearer {API_KEY}",

    "Content-Type": "application/json"
}

# =========================================================
# 16. FUNCION AUXILIAR
# =========================================================

def construir_ejemplos(strategy):

    ejemplos = ""

    lista = PROMPT_EXAMPLES[strategy]["ejemplos"]

    for ej in lista:

        ejemplos += (
            f"Texto: {ej['texto']}\n"
            f"Respuesta: {ej['label']}\n\n"
        )

    return ejemplos
# =========================================================
# 17. DEEPSEEK PREDICT MULTICLASE
# =========================================================

CLASSES = CLASSES = list(label_encoder.classes_)

def deepseek_predict_proba(
    texto,
    strategy="cot"
):

    ejemplos = construir_ejemplos(strategy)

    prompt = f"""
Eres un clasificador experto de textos cientificos.

Debes clasificar el texto en UNA de estas 8 clases:

CONTR = contribución o aportes
INTRO = introduccion del tema
BACK = antecedentes o marco teorico
METHO = metodologia
LIM = limitaciones
RES = resultados
CONC = conclusiones
DISC = discusion

Ejemplos:

{ejemplos}

Texto:
{texto}

Responde SOLO en JSON con probabilidades
que sumen 1.

Formato EXACTO:

{{
"CONTR":0.xx,
"INTRO":0.xx,
"BACK":0.xx,
"METHO":0.xx,
"LIM":0.xx,
"RES":0.xx,
"CONC":0.xx,
"DISC":0.xx
}}
"""

    payload = {
        "model": "deepseek-chat",
        "messages": [
            {
                "role": "system",
                "content": "Eres un clasificador cientifico."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        "temperature": 0.0
    }

    try:

        response = requests.post(
            API_URL,
            headers=headers,
            json=payload,
            timeout=60
        )

        result = response.json()

        salida = result["choices"][0]["message"]["content"]

        data_json = json.loads(salida)

        probs = np.array([
            float(data_json.get(c, 0))
            for c in CLASSES
        ])

        # normalizar seguridad
        if probs.sum() == 0:
            probs = np.ones(len(CLASSES)) / len(CLASSES)
        else:
            probs = probs / probs.sum()

        return probs

    except Exception as e:

        print(e)

        return np.ones(len(CLASSES)) / len(CLASSES)

# =========================================================
# 18. ENSAMBLE HIBRIDO (MULTICLASE)
# =========================================================
def hybrid_predict(
    texto,
    peso_lgbm=0.8,
    peso_deepseek=0.2,
    strategy="cot"
):

    X_text = vectorizer.transform([texto])
    X_input = sp.hstack([X_text])

    # LightGBM probs
    prob_lgbm = lgb_model.predict_proba(X_input)[0]

    # DeepSeek probs
    prob_deepseek = deepseek_predict_proba(
        texto,
        strategy=strategy
    )

    # Ensemble
    prob_final = (
        peso_lgbm * prob_lgbm +
        peso_deepseek * prob_deepseek
    )

    prob_final = prob_final / prob_final.sum()

    # devolver índice numérico
    pred_final = int(np.argmax(prob_final))

    return {
        "pred_final": pred_final,
        "prob_lgbm": prob_lgbm,
        "prob_deepseek": prob_deepseek,
        "prob_final": prob_final
    }

# =========================================================
# 19. EVALUACION HIBRIDA EVAL
# =========================================================

print("\n===================================")
print("EVALUACION HIBRIDA EVAL")
print("===================================")

# etiquetas reales (YA NUMERICAS)
y_true = eval_data["label"].tolist()

preds_hybrid = []

for i, texto in enumerate(eval_data["text"]):

    print(f"Procesando {i+1}/{len(eval_data)}")

    resultado = hybrid_predict(
        texto=texto,
        peso_lgbm=0.8,
        peso_deepseek=0.2,
        strategy="cot"
    )

    # ahora pred_final es NUMERICO
    preds_hybrid.append(resultado["pred_final"])

    time.sleep(1)  # evita rate limit API


# =========================================================
# 20, RESULTADOS HIBRIDOS
# =========================================================

evaluar_modelo(
    "Hybrid LightGBM + DeepSeek",
    y_true,
    preds_hybrid
)

# =========================================================
# 21. EJEMPLO INDIVIDUAL
# =========================================================

resultado = hybrid_predict(

    texto="""
    los resultados contradicen la teoria principal
    y no se encontraron mejoras significativas
    """,

    strategy="cot"
)

print("\n===================================")
print("EJEMPLO PREDICCION")
print("===================================")

print(resultado)

# =========================================================
# 22. COMPARACION CONCEPTUAL
# =========================================================

print("\n================================================")
print("COMPARACION CONCEPTUAL")
print("================================================")

print("""
LightGBM:
Modelo estadistico supervisado basado en boosting.
Excelente para datasets tabulares y TF-IDF.
""")

print("""
DeepSeek Prompting:
Modelo semantico basado en lenguaje natural.
Capaz de detectar contradicciones implicitas.
""")

print("""
Sistema Hibrido:
Combina:
- precision estadistica
- comprension semantica

LightGBM aporta robustez numerica.
DeepSeek aporta comprension contextual.
""")

print("""
Este enfoque corresponde a:

LLM-Augmented Machine Learning
o
Hybrid Neuro-Symbolic AI
""")


EVALUACION HIBRIDA EVAL
Procesando 1/1699


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Procesando 2/1699
Procesando 3/1699
Procesando 4/1699
Procesando 5/1699
Procesando 6/1699
Procesando 7/1699
Procesando 8/1699
Procesando 9/1699
Procesando 10/1699
Procesando 11/1699
Procesando 12/1699
Procesando 13/1699
Procesando 14/1699
Procesando 15/1699
Procesando 16/1699
Procesando 17/1699
Procesando 18/1699
Procesando 19/1699
Procesando 20/1699
Procesando 21/1699
Procesando 22/1699
Procesando 23/1699
Procesando 24/1699
Procesando 25/1699
Procesando 26/1699
Procesando 27/1699
Procesando 28/1699
Procesando 29/1699
Procesando 30/1699
Procesando 31/1699
Procesando 32/1699
Procesando 33/1699
Procesando 34/1699
Procesando 35/1699
Procesando 36/1699
Procesando 37/1699
Procesando 38/1699
Procesando 39/1699
Procesando 40/1699
Procesando 41/1699
Procesando 42/1699
Procesando 43/1699
Procesando 44/1699
Procesando 45/1699
Procesando 46/1699
Procesando 47/1699
Procesando 48/1699
Procesando 49/1699
Procesando 50/1699
Procesando 51/1699
Procesando 52/1699
Procesando 53/1699
Procesando 54/1699
P

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



EJEMPLO PREDICCION
{'pred_final': 7, 'prob_lgbm': array([0.00383779, 0.00815101, 0.00292527, 0.0057537 , 0.05922947,
       0.00236607, 0.02230567, 0.89543103]), 'prob_deepseek': array([0.01010101, 0.        , 0.95959596, 0.        , 0.01010101,
       0.01010101, 0.        , 0.01010101]), 'prob_final': array([0.00509043, 0.00652081, 0.19425941, 0.00460296, 0.04940378,
       0.00391306, 0.01784453, 0.71836503])}

COMPARACION CONCEPTUAL

LightGBM:
Modelo estadistico supervisado basado en boosting.
Excelente para datasets tabulares y TF-IDF.


DeepSeek Prompting:
Modelo semantico basado en lenguaje natural.
Capaz de detectar contradicciones implicitas.


Sistema Hibrido:
Combina:
- precision estadistica
- comprension semantica

LightGBM aporta robustez numerica.
DeepSeek aporta comprension contextual.


Este enfoque corresponde a:

LLM-Augmented Machine Learning
o
Hybrid Neuro-Symbolic AI

